# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
!pip uninstall transformers
!pip install transformers==4.57.1

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Would remove:
    /usr/local/bin/transformers
    /usr/local/lib/python3.12/dist-packages/transformers-5.0.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/transformers/*
Proceed (Y/n)? Y
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1


In [2]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [3]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.6 MB/s eta 0:00:00


In [4]:
import os
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate


In [5]:
import os
from datetime import datetime
import ipykernel
import requests
import json

class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    MODEL_NAME = "google/byt5-small"

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 10
    LEARNING_RATE = 2e-4

    # Dynamically determine the output directory based on notebook path and timestamp
    try:
        connection_file = os.path.basename(ipykernel.get_connection_file())
        kernel_id = connection_file.split('-', 1)[1].split('.')[0]
        response = requests.get('http://172.28.0.12:9000/api/sessions')
        sessions = json.loads(response.text)
        notebook_path = "unknown_notebook.ipynb" # Default in case of failure
        for session in sessions:
            if session['kernel']['id'] == kernel_id:
                notebook_path = session['path']
                break
        notebook_dir = os.path.dirname(notebook_path)
        notebook_name = os.path.splitext(os.path.basename(notebook_path))[0]
    except Exception:
        notebook_dir = "." # Fallback to current working directory
        notebook_name = "colab_notebook_run" # Fallback notebook name

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = os.path.join(notebook_dir, f"{notebook_name}_{timestamp}")

    # ============ CV (5-fold) ============
    CV_N_SPLITS = 5
    CV_SEED = 42
    USE_GROUP_KFOLD = True
    SAVE_EACH_FOLD_MODEL = True

    # Save fold artifacts under OUTPUT_DIR
    FOLD_OUTPUT_ROOT = f"{OUTPUT_DIR}/cv5"
    REPORT_CSV = "cv_results.csv"

In [6]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything()

In [7]:
INPUT_DIR = "/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge/data"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [8]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [9]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [10]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

# Prepare 5-fold CV splits.
cv_df = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(cv_df.columns)

groups = cv_df["oare_id"].values
X_idx = np.arange(len(cv_df))

splitter = GroupKFold(n_splits=Config.CV_N_SPLITS)
fold_splits = list(splitter.split(X_idx, y=None, groups=groups))

print(f"Prepared {len(fold_splits)} folds (GroupKFold by oare_id)")


Expanded Train Data: 1561 sentences (Alignment applied)
Prepared 5 folds (GroupKFold by oare_id)


In [11]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK2EN = "translate Akkadian to English: "
PREFIX_EN2AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Unify common gap markers to a single token for better pattern consistency.
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_akk2en_input(transliteration: str) -> str:
    return PREFIX_AKK2EN + normalize_transliteration(transliteration)


def build_en2akk_input(translation: str) -> str:
    return PREFIX_EN2AKK + ("" if translation is None else str(translation))


def create_bidirectional_dataset(ds_split, seed: int = 42):
    """Make a 2x dataset: (akk->en) + (en->akk).

    This follows the idea in `notebooks/004/dpc-baseline-train-infer.ipynb`:
    keep task-direction as a natural-language prefix in the input.
    """
    df = ds_split.to_pandas()

    # Direction 1: Akkadian -> English (main task)
    df_fwd = df.copy()
    df_fwd["input_text"] = [build_akk2en_input(x) for x in df_fwd["transliteration"].tolist()]
    df_fwd["target_text"] = df_fwd["translation"].astype(str)

    # Direction 2: English -> Akkadian (reverse task)
    df_bwd = df.copy()
    df_bwd["input_text"] = [build_en2akk_input(x) for x in df_bwd["translation"].tolist()]
    df_bwd["target_text"] = [normalize_transliteration(x) for x in df_bwd["transliteration"].tolist()]

    df_combined = pd.concat([df_fwd, df_bwd], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=seed).reset_index(drop=True)

    return Dataset.from_pandas(df_combined)


def create_unidirectional_dataset(ds_split):
    """Validation dataset is kept as Akkadian -> English only for evaluation."""
    df = ds_split.to_pandas()
    df["input_text"] = [build_akk2en_input(x) for x in df["transliteration"].tolist()]
    df["target_text"] = df["translation"].astype(str)
    return Dataset.from_pandas(df)


def preprocess_function(examples):
    inputs = [str(ex) for ex in examples["input_text"]]
    targets = [str(ex) for ex in examples["target_text"]]

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [12]:
# ==========================================
# 4. 5-fold CV training (fine-tuning)
# ==========================================

# Metrics: competition uses geo mean of BLEU and chrF++ (corpus-level)
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Sometimes Seq2SeqTrainer can return logits; convert them to token ids.
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    # ByT5 decoding can crash if ids are negative/out of vocab.
    preds = np.asarray(preds)
    preds = preds.astype(np.int64, copy=False)
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    refs = [[x] for x in decoded_labels]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=refs)["score"]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=refs)["score"]

    geo_mean = (bleu * chrf) ** 0.5 if (bleu > 0 and chrf > 0) else 0.0

    return {"chrf": chrf, "bleu": bleu, "geo_mean": geo_mean}


# Build a single HF dataset and slice by fold indices.
ds_all = Dataset.from_pandas(cv_df)

fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print("" + "=" * 60)
    print(f"FOLD {fold}/{Config.CV_N_SPLITS - 1}")
    print("=" * 60)

    ds_train_raw = ds_all.select(tr_idx.tolist())
    ds_val_raw = ds_all.select(va_idx.tolist())

    # Train is bidirectional (akk->en + en->akk): 2x samples
    ds_train = create_bidirectional_dataset(ds_train_raw, seed=int(Config.CV_SEED) + int(fold))
    # Validation stays unidirectional (akk->en) for metric consistency
    ds_val = create_unidirectional_dataset(ds_val_raw)

    # Tokenize inside fold
    tokenized_train = ds_train.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_train.column_names,
    )
    tokenized_val = ds_val.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_val.column_names,
    )

    # Fresh model per fold
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    fold_out = f"{Config.FOLD_OUTPUT_ROOT}/fold_{fold}"
    fold_model_out = f"{fold_out}/model"
    os.makedirs(fold_out, exist_ok=True)

    args = Seq2SeqTrainingArguments(
        output_dir=fold_out,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=Config.LEARNING_RATE,

        # === Key fixes ===
        fp16=False,                    # FP16 can be unstable for some setups; keep FP32 by default.
        bf16=True,              # A100ならまずこれを試す
        tf32=True,              # 速度目的（メモリはほぼ変わらない）
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        generation_max_length=Config.MAX_LENGTH,  # Crucial for ByT5; avoid too-short defaults
        generation_num_beams=2,
        # ==================

        weight_decay=0.01,
        num_train_epochs=Config.EPOCHS,
        predict_with_generate=True,
        logging_strategy="epoch",
        logging_steps=10,
        disable_tqdm=True,
        report_to="none",
        load_best_model_at_end=False,
    )

    traitner = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    print("Starting Training (fold CV, FP32 mode)...")
    trainer.train()

    eval_metrics = trainer.evaluate()
    fold_result = {
        "fold": int(fold),
        "n_train": int(len(tokenized_train)),
        "n_val": int(len(tokenized_val)),
        "geo_mean": float(eval_metrics.get("eval_geo_mean", float("nan"))),
        "bleu": float(eval_metrics.get("eval_bleu", float("nan"))),
        "chrf": float(eval_metrics.get("eval_chrf", float("nan"))),
        "fold_model_path": fold_model_out,
    }
    fold_metrics.append(fold_result)

    print("Fold metrics:", fold_result)

    if Config.SAVE_EACH_FOLD_MODEL:
        trainer.save_model(fold_model_out)
        tokenizer.save_pretrained(fold_model_out)
        print(f"Saved fold model to: {fold_model_out}")

    # Explicit cleanup to avoid memory/disk pressure across folds
    del trainer, model, data_collator, tokenized_train, tokenized_val, ds_train, ds_val
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summarize CV
cv_results = pd.DataFrame(fold_metrics)
report_path = f"{Config.FOLD_OUTPUT_ROOT}/{Config.REPORT_CSV}"
os.makedirs(Config.FOLD_OUTPUT_ROOT, exist_ok=True)
cv_results.to_csv(report_path, index=False)

print("" + "=" * 60)
print("5-FOLD CV SUMMARY")
print("=" * 60)
print(cv_results[["fold", "n_train", "n_val", "geo_mean", "bleu", "chrf", "fold_model_path"]])

mean_score = cv_results["geo_mean"].mean()
std_score = cv_results["geo_mean"].std(ddof=1)
print(f"geo_mean: mean={mean_score:.4f}, std={std_score:.4f}")

best_i = int(cv_results["geo_mean"].idxmax())
print(f"Best fold index: {best_i}")
print(f"Best fold model path: {cv_results.loc[best_i, 'fold_model_path']}")
print(f"Saved CV report CSV to: {report_path}")


FOLD 0/4


Map:   0%|          | 0/2496 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

Starting Training (fold CV, FP32 mode)...
{'loss': 1.9311, 'grad_norm': 1.1292005777359009, 'learning_rate': 0.00018025316455696203, 'epoch': 1.0}
{'eval_loss': 0.9009466171264648, 'eval_chrf': 10.278324854239992, 'eval_bleu': 0.6946409013528716, 'eval_geo_mean': 2.6720300973504942, 'eval_runtime': 164.8824, 'eval_samples_per_second': 1.892, 'eval_steps_per_second': 0.121, 'epoch': 1.0}
{'loss': 1.0422, 'grad_norm': 1.3359692096710205, 'learning_rate': 0.00016025316455696204, 'epoch': 2.0}
{'eval_loss': 0.770791232585907, 'eval_chrf': 17.721251815599604, 'eval_bleu': 2.8028218097956166, 'eval_geo_mean': 7.047659972334274, 'eval_runtime': 163.1111, 'eval_samples_per_second': 1.913, 'eval_steps_per_second': 0.123, 'epoch': 2.0}
{'loss': 0.9276, 'grad_norm': 3.1213908195495605, 'learning_rate': 0.00014025316455696204, 'epoch': 3.0}
{'eval_loss': 0.7201955914497375, 'eval_chrf': 23.345022639505224, 'eval_bleu': 5.771791547856402, 'eval_geo_mean': 11.607868208900918, 'eval_runtime': 161.622

Map:   0%|          | 0/2498 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

/tmp/ipykernel_575/812782177.py:111: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (fold CV, FP32 mode)...
{'loss': 1.9356, 'grad_norm': 3.7548153400421143, 'learning_rate': 0.00018025316455696203, 'epoch': 1.0}
{'eval_loss': 0.9381322860717773, 'eval_chrf': 9.68767108739857, 'eval_bleu': 0.39699697284862184, 'eval_geo_mean': 1.9611160331939437, 'eval_runtime': 164.7797, 'eval_samples_per_second': 1.893, 'eval_steps_per_second': 0.121, 'epoch': 1.0}
{'loss': 1.0402, 'grad_norm': 0.9196438789367676, 'learning_rate': 0.00016025316455696204, 'epoch': 2.0}
{'eval_loss': 0.779448926448822, 'eval_chrf': 16.31316035286858, 'eval_bleu': 3.046366585129252, 'eval_geo_mean': 7.049529530176759, 'eval_runtime': 163.0837, 'eval_samples_per_second': 1.913, 'eval_steps_per_second': 0.123, 'epoch': 2.0}
{'loss': 0.9091, 'grad_norm': 1.279389500617981, 'learning_rate': 0.00014025316455696204, 'epoch': 3.0}
{'eval_loss': 0.7236851453781128, 'eval_chrf': 23.14722484338682, 'eval_bleu': 5.552917605972384, 'eval_geo_mean': 11.33731151385742, 'eval_runtime': 162.3003, 'ev

In [13]:
# --- Notes ---
# Fold models are saved under:
#   ./byt5-akkadian-model/cv5/fold_{k}/model
# The CV summary is saved to:
#   ./byt5-akkadian-model/cv5/cv_results.csv
